# STEP 4-C — 크롭 비교 (m1.5 vs m2.5)

## 왜 지금 이걸 하나

증강(augmentation)은 닫혔고 해상도는 384 가 상한입니다. 그런데 **전처리 축을
하나 안 열어봤습니다** — 크롭 창 크기입니다.

크롭 감사(crop audit)에 이 숫자가 있습니다:

```
크롭 짧은 변 중앙값 242px
384px 미만 = 확대해서 씀: 74.6%
```

**크롭의 4분의 3이 없는 픽셀을 만들어내고 있습니다.** 병변이 원본에서 작으니
`m1.5` 로 자르면 창이 작고, 그걸 384 로 늘리는 겁니다.

클래스별로 계산해 보면:

| 클래스 | 원본 박스 | m1.5 창 | 확대 | m2.5 창 | 확대 |
|---|---:|---:|---:|---:|---:|
| A1 구진·플라크 | 99px | 148px | **2.6x** | 247px | 1.6x |
| A2 비듬·각질 | 160px | 240px | 1.6x | 399px | 1.0x |
| A3 태선화 | 174px | 261px | 1.5x | 435px | 0.9x |
| A4 농포·여드름 | 163px | 244px | 1.6x | 407px | 0.9x |
| A5 미란·궤양 | 155px | 233px | 1.7x | 388px | 1.0x |
| A6 결절·종괴 | 253px | 379px | 1.0x | 632px | 0.6x |

**A1 이 가장 심하게 늘어납니다(2.6배).** 그리고 A1 은 하필
shortcut baseline 대비 격차가 **+0.047 뿐**인 클래스입니다 — 사진에서 거의
못 배우고 있는데, 그 사진이 흐릿하게 늘려진 것이었을 수 있습니다.

## 무엇을 비교하나

같은 병변을 **얼마나 넓게 자를지**만 바꿉니다.

| 크롭 | 병변이 384 입력에서 | 화질 | 주변 맥락 |
|---|---|---|---|
| **`m1.5`** (현재) | 크게 (2/3 차지) | **흐림** — 늘린 픽셀 | 적음 |
| **`m2.5`** | 작게 (2/5 차지) | 선명 — 원본 픽셀 | 많음 |

**"크게 보되 흐리게" vs "작게 보되 선명하게".**
어느 쪽이 나은지는 돌려봐야 압니다.

## 이걸 먼저 하는 이유

크롭은 다른 축과 성격이 다릅니다. 증강이나 백본을 바꾸면 **모델만** 바뀌지만,
크롭을 바꾸면 **입력 자체가 바뀝니다.** 그러면 그 뒤에 정하는 것들
— 촬영 가이드(capture guideline), 임계값(threshold), 백본 순위 — 이 **전부 달라집니다.**

기준이 흔들리는 상태에서 뒤를 정하면 다시 해야 합니다. 그래서 여기가 먼저입니다.

## 판정 기준 (돌리기 **전에** 정해둡니다)

| m2.5 결과 | 판정 |
|---|---|
| macro-F1 +0.02 이상 **또는** 배율 하락 3%p 이상 감소 | ✅ m2.5 채택 |
| 둘 다 잡음(±0.01 / ±3%p) 안 | ➖ m1.5 유지 (확대는 문제가 아니었음) |
| 뚜렷하게 나쁨 | ❌ m1.5 유지. 맥락보다 병변 크기가 중요하다는 뜻 |

**A1 recall 을 특히 보세요.** 확대가 원인이었다면 A1 이 가장 크게 움직입니다.
전체 macro-F1 은 그대로인데 A1 만 오르는 경우도 의미가 있습니다.

⚠️ 이번엔 **서브셋을 안 씁니다.** 입력이 바뀌는 실험이라 확실한 답이 필요하고,
2종이면 풀 데이터로도 한 세션에 들어갑니다.

⏱️ 2종 × 약 78분 ≈ **2시간 40분**. `Save & Run All (Commit)`.

## 준비물

Kaggle 입력에 데이터셋 **3개**가 붙어 있어야 합니다:
`dogskin-full` · `dogskin-m15` · **`dogskin-m25`**

없으면 로컬에서:
```
uv run python prepare_local.py --package --tags m2.5 --out dogskin_m25.zip
```


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

_PKGS = ["timm", "imagehash", "pyarrow", "grad-cam", "albumentations"]
_ok = False
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"],
                  check=False).returncode == 0:
    _ok = subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q",
                          "--system", *_PKGS], check=False).returncode == 0
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_PKGS], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-22.3"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


---
## 1. 데이터 붙이기


In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ── 다른 환경에서 학습한 체크포인트 가져오기 (Colab → Kaggle 이주) ──────
#    Colab 에서 이미 학습을 끝냈다면, Drive 의 dogskin_work/checkpoints 를
#    Kaggle 데이터셋으로 올린 뒤 그 경로를 여기에 주세요.
#    가져온 실험은 '완료' 로 인식되어 학습 셀이 ⏭️ 로 건너뜁니다.
#
# train.import_checkpoints("/kaggle/input/dogskin-ckpt")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages, experiments
from src.config import CLASSES

env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 384      # 03 해상도 실험에서 채택
EPOCHS   = 25       # 03 의 2단계와 동일 (epoch 17 에서 조기 종료됐습니다)
CROPS    = ("m1.5", "m2.5")

# 03 풀 실행 실측 (m1.5 / 384 / 표본 2,000장) — 이번 m1.5 가 여기서 크게
# 벗어나면 데이터·환경이 달라진 것이므로 먼저 원인을 찾으세요.
BASE_M15 = {"macro_f1": 0.5697, "scale_drop": 0.259, "a6_recall": 0.405}

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
have = crop.available_tags()
print(f"사용 가능한 크롭 태그: {have}")
missing = [t for t in CROPS if t not in have]
if missing:
    raise SystemExit(
        f"❌ 크롭 태그 {missing} 가 없습니다.\n"
        f"   Kaggle 우측 Add Input 에서 dogskin-m25 를 붙였는지 확인하세요.\n"
        f"   로컬에서 만들려면: uv run python prepare_local.py --package --tags m2.5")

# ── ★ 학습 **전에** 두 크롭을 모두 검증합니다 ────────────────────
# ⚠️ switch_tag 은 커버리지 95% 미만이면 에러를 냅니다. 그 호출이 학습 루프
#    안에 있으면, m2.5 업로드가 덜 끝난 경우 **m1.5 를 78분 학습한 뒤에** 터집니다.
#    실제로 full 크롭이 30% 만 올라갔던 적이 있습니다. 여기서 미리 걸러냅니다.
views = {}
for _t in CROPS:
    print(f"\n[준비] '{_t}' 확인")
    _d = crop.switch_tag(df, _t)                     # 커버리지 미달이면 여기서 멈춤
    _v = stages.to_stage2(_d)
    split.verify(_v, fold=0, strict=True)            # 크롭을 바꿔도 누수(leakage)는 없어야
    views[_t] = _v
    _tr, _va = split.get_fold(_v, 0)
    print(f"  → 2단계 {len(_v):,}행  (train {len(_tr):,} / val {len(_va):,})")

_sizes = {t: len(v) for t, v in views.items()}
if len(set(_sizes.values())) > 1:
    raise SystemExit(f"❌ 크롭마다 행 수가 다릅니다 {_sizes} — 같은 매니페스트라 같아야 합니다")
print(f"\n✅ 두 크롭 모두 준비됨. 이제 학습을 시작해도 됩니다 ({_sizes})")


---
## 2. 크롭 2종 학습·비교


In [ ]:
# 위에서 이미 검증·생성한 view 를 씁니다 (여기서 실패할 일이 없어야 합니다)
runs = []
for tag in CROPS:
    runs.append(experiments.train_and_measure(
        views[tag], stage=2, img_size=IMG_SIZE, crop_tag=tag,
        device=DEV, epochs=EPOCHS, n_robust=2000))

verdict = experiments.crop_report(runs, baseline="m1.5")


---
## 3. 클래스별로 보기 + 판정


In [ ]:
# ★ 클래스별로 봅니다 — 확대가 원인이었다면 A1 이 가장 크게 움직입니다
import json

print("=" * 66)
print(" STEP 4C 크롭 비교 (이 블록을 복사해서 공유하세요)")
print("=" * 66)
print(f"  {'크롭':<8}{'macro-F1':>10}{'배율하락':>10}{'A6 recall':>11}{'A1 recall':>11}")
for r in runs:
    pc = r["report"].metrics["per_class"]["recall"]
    _dr = r.get('scale_drop')
    print(f"  {r['crop_tag']:<8}{r['score']:>10.4f}"
          f"{('  못 잼' if _dr is None else f'{_dr:>9.1%}')}"
          f"{pc[CLASSES.index('A6')]:>11.3f}{pc[CLASSES.index('A1')]:>11.3f}")

ADOPTED = "m1.5"        # 판정에서 뒤집히지 않으면 기준 크롭 유지
base = next(r for r in runs if r["crop_tag"] == "m1.5")
alt  = next((r for r in runs if r["crop_tag"] != "m1.5"), None)
if alt:
    bpc = base["report"].metrics["per_class"]["recall"]
    apc = alt["report"].metrics["per_class"]["recall"]
    print(f"\n  클래스별 변화 (m1.5 → {alt['crop_tag']})")
    for i, c in enumerate(CLASSES):
        d = apc[i] - bpc[i]
        mark = "  ←" if abs(d) >= 0.05 else ""
        print(f"    {c}  {bpc[i]:.3f} → {apc[i]:.3f}  ({d:+.3f}){mark}")

    d_f1 = alt["score"] - base["score"]
    # 배율 하락은 _summary 가 없으면 None 일 수 있습니다 (기준 조건 누락 등)
    d_dr = (alt["scale_drop"] - base["scale_drop"]
            if alt.get("scale_drop") is not None and base.get("scale_drop") is not None
            else None)
    print(f"\n  판정 기준: macro-F1 +0.02 이상 또는 배율 하락 3%p 이상 감소 → 채택")
    print(f"    macro-F1  {d_f1:+.4f}")
    print(f"    배율 하락  {'못 잼' if d_dr is None else f'{d_dr:+.1%}'}")
    if d_f1 >= 0.02 or (d_dr is not None and d_dr <= -0.03):
        ADOPTED = alt["crop_tag"]
        print(f"    ✅ {ADOPTED} 채택 — 확대가 실제로 발목을 잡고 있었습니다")
    elif d_f1 <= -0.02 or (d_dr is not None and d_dr >= 0.03):
        print("    ❌ m1.5 유지 — 맥락보다 병변 크기가 중요합니다")
    else:
        print("    ➖ m1.5 유지 — 확대는 병목이 아니었습니다 (둘 다 잡음 안)")

# ★ 05 가 이걸 읽어서 크롭을 정합니다. 문자열로 남기지 않으면 05 가 다시 추측해야 합니다.

print(f"\n  03 기준(m1.5): macro-F1 {BASE_M15['macro_f1']:.4f} / "
      f"하락 {BASE_M15['scale_drop']:.1%} / A6 {BASE_M15['a6_recall']:.3f}")
print("=" * 66)

W = env.work_root(); (W/"reports").mkdir(parents=True, exist_ok=True)
keep = ("stage","img_size","crop_tag","exp_name","epochs","batch_size","minutes",
        "best_epoch","n_epochs","converged","score","score_name","macro_f1",
        "a6_recall","scale_drop","scale_worst","scale_worst_at")
(W/"reports"/"step4c_crop.json").write_text(json.dumps({
    "img_size": IMG_SIZE, "epochs": EPOCHS, "baseline_03": BASE_M15,
    "best_crop": ADOPTED,
    "runs": [{k: r[k] for k in keep if k in r} for r in runs],
    "per_class_recall": {r["crop_tag"]: dict(zip(
        CLASSES, r["report"].metrics["per_class"]["recall"])) for r in runs},
    "verdict": {s: v.get("verdict") for s, v in verdict.items()},
}, indent=2, ensure_ascii=False))
print(f"저장: {W/'reports'/'step4c_crop.json'}")


---
## 다음 단계

| 이번 결과 | 다음 |
|---|---|
| ✅ m2.5 채택 | 03 의 `BEST_CROP` 을 바꾸고 베이스라인 재측정 → 그 뒤 05 |
| ➖ / ❌ m1.5 유지 | 전처리 축도 닫힙니다 → 바로 `05` (촬영 가이드 + 보정·임계값) |

어느 쪽이든 **전처리는 여기서 끝**입니다. 남은 자동 실험은:

1. `05` — 촬영 가이드(capture guideline) 측정 + 확률 보정(calibration) +
   클래스별 임계값. **학습 0회**이고 A6 recall 0.405 를 직접 겨냥합니다
2. `04` — 백본(backbone) 6종 비교

그 다음이 **정성평가(qualitative evaluation)** 입니다 (멘토 피드백 5번).
자동으로 짜낼 수 있는 걸 다 짜낸 뒤에 손을 댑니다.

📖 [`docs/멘토링_피드백.md`](../docs/멘토링_피드백.md) ·
[`docs/results/STEP4B_증강스윕_실측.md`](../docs/results/STEP4B_증강스윕_실측.md)
